In [1]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


# client = OpenAI()


/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [3]:

df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

df["Errors?"] = (df["Errors?"] != "None").astype(int)

df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

df = df.fillna(0)

# print(df.dtypes)
# print(df.head())

/tmp/ipykernel_4106319/3282332532.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [4]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [5]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

## Run form here

In [6]:
# # STEP 4: Connect to Ollama
# import requests

# def call_llm(prompt):
#     response = requests.post(
#         "http://localhost:11434/api/generate",
#         json={
#             "model": "llama3",
#             "prompt": prompt,
#             "stream": False
#         }
#     )

#     data = response.json()
    
#     if "response" in data:
#         return data["response"]
#     else:
#         print("Error:", data)
#         return None

In [7]:

from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/tmp/ipykernel_4106319/2009901296.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [8]:
for m in genai.list_models():
    print(m.name, "->", m.supported_generation_methods)

models/gemini-2.5-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts -> ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts -> ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it -> ['generateContent', 'countTokens']
models/gemma-3-4b-it -> ['generateContent', 'countTokens']
models/gemma-3-12b-it -> ['generateContent

In [9]:
model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [10]:

# df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# df["Errors?"] = (df["Errors?"] != "None").astype(int)

# df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# df = df.fillna(0)

In [11]:
# PROMPT = """
# You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds —  directly)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded —  directly)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a Classifier detect fraud.

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users


# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features

# IMPORTANT RULES:

# - Always use "Hour" for time-based features (NOT Time)
# - Merchant City / State → ONLY nunique or count
# - Ensure all outputs are numeric scalars per user

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# Generate 12–15 high-quality fraud detection features.

# Focus on:
# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior
# - error behavior
# - chip usage patterns

# Do NOT compute values.
# Do NOT output anything outside JSON.


# """

In [ ]:
Prompt = """
==============================
DATA SUMMARY
==============================

- Number of users: 2000
- Transactions span: multiple years (1991–2020)
- Amount has high variability (~100K unique values)
- Merchant Name is high-cardinality (~100K unique)
- Merchant City is high-cardinality (~13K)
- Merchant State includes global locations (223 unique)
- Errors column contains multiple error types (including combinations)
- Fraud label is binary (Yes/No)

==============================
VALUE DISTRIBUTION INSIGHTS
==============================

Use Chip:
- Swipe Transaction
- Online Transaction
- Chip Transaction

Errors?:
- NaN (no error)
- Technical Glitch
- Insufficient Balance
- Bad PIN
- Bad CVV
- Bad Zipcode
- Combinations of above errors

Merchant State:
- Mix of US states and international locations

Is Fraud:
- Yes / No

==============================
SAMPLE DATA (STRUCTURE)
==============================

Each row represents a transaction:

User=0, Card=0, Year=2002, Month=9, Day=1, Time=06:21,
Amount=$134.09, Use Chip=Swipe Transaction,
Merchant Name=high-cardinality ID,
Merchant City=La Verne, Merchant State=CA,
Zip=91750, MCC=5300, Errors?=NaN, Is Fraud=No

==============================
PREPROCESSING & MISSING VALUE HANDLING
==============================

Before feature computation, assume the following preprocessing has been applied:

1. Amount:
   - Remove "$" and convert to float
   - Missing values (if any) → fill with median Amount

2. Time:
   - Convert "HH:MM" → Hour (0–23)
   - Missing values → fill with mode Hour

3. Use Chip:
   - Encode as:
       Swipe Transaction → 0
       Chip Transaction → 1
       Online Transaction → 2
   - Missing values → fill with most frequent category

4. Errors?:
   - NaN → 0 (no error)
   - Any error string → 1 (error present)

5. Merchant City / Merchant State:
   - Missing values → fill with "Unknown"

6. Zip:
   - Missing values → fill with median Zip

7. MCC:
   - Missing values → fill with mode MCC

8. Is Fraud:
   - Yes → 1, No → 0
   - MUST NOT be used for feature generation

After preprocessing, the following cleaned columns are available:

- Amount (float)
- Hour (int)
- Use Chip (int)
- Errors? (int)
- MCC (int)
- Merchant City (string)
- Merchant State (string)
- Zip (float)


==============================
IMPORTANT DATA CHARACTERISTICS
==============================

- Merchant Name is extremely high-cardinality → avoid direct usage
- Merchant City/State have high diversity → use aggregation (nunique)
- Time is granular → useful for behavioral patterns
- Errors column contains multiple combined error types
- Amount has high variance → useful for anomaly detection



============================== 
YOUR TASK 
============================== 
Design USER-LEVEL features for fraud detection. Each feature MUST produce ONE value per User.

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

Return a JSON list:

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "intuition": "...",
    "type": "numeric"
  }
]

- pandas_code must be executable
- must return a Series indexed by User
- no explanations outside JSON

==============================
EXECUTION REQUIREMENTS
==============================

- Features must be computed using pandas
- Final output must be a Series indexed by User
- Length must equal number of unique users
- Must not use undefined variables
- Must not use target column (Is Fraud)

==============================
ALLOWED OPERATIONS
==============================

You may use:

- multiple aggregations
- ratios (e.g., sum / count)
- differences (max - mean)
- standard deviation and variance
- conditional aggregations
- arithmetic combinations of features

==============================
QUALITY CONSTRAINTS
==============================

- Avoid redundant or highly correlated features
- Prefer interpretable features
- Prefer features that capture behavioral differences
- Avoid trivial features (e.g., constant or near-constant)

==============================
FINAL INSTRUCTIONS
==============================

- Generate 12–15 features
- Each feature must be unique and meaningful
- Output ONLY JSON
- Do NOT include explanations outside JSON

Ensure final output remains a valid Series per User.
"""

In [12]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

The features you generate will be used to train a RandomForestClassifier.

==============================
DATASET SCHEMA (STRICT)
==============================

- User → int64 (group key)
- Card → int64 
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
- Hour → float64 (0–23, derived from Time — USE THIS)
- Amount → float64 (numeric)
- Use Chip → float64 (1 = Yes, 0 = No)
- Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
- Merchant City → object (categorical — ONLY use nunique/count)
- Merchant State → object (categorical — ONLY use nunique/count)
- Zip → float64 (numeric)
- MCC → int64 (category code — can use nunique/count)
- Errors? → int64 (1 = error, 0 = no error)
- Is Fraud? → int64 (target label — DO NOT USE)

==============================
YOUR TASK
==============================

Design USER-LEVEL features for fraud detection.

Each feature MUST produce ONE value per User.

==============================
STRICT REQUIREMENTS (VERY IMPORTANT)
==============================

- Features must be NUMERIC only (int or float)
- Must be directly usable in sklearn (no preprocessing needed)
- MUST be computed ONLY from df
- MUST NOT depend on other generated features
- SAME features for ALL users

FEATURE DESIGN FOCUS:

- spending patterns
- transaction frequency
- merchant/category diversity
- geographic spread
- time behavior (USE Hour only)
- error behavior
- chip usage patterns

==============================
CRITICAL RULE (VERY IMPORTANT)
==============================

- Each feature must apply ONLY ONE aggregation
- After df.groupby('User'), apply EXACTLY ONE aggregation
- DO NOT apply aggregation on top of aggregation

INVALID:
df.groupby('User')['Amount'].mean().mean()
df.groupby('User')['MCC'].nunique().mean()
df.groupby('User').size().mean()

VALID:
df.groupby('User')['Amount'].mean()
df.groupby('User')['MCC'].nunique()
df.groupby('User').size()

If the result is a single number (scalar), the feature is INVALID.

==============================
CRITICAL EXECUTION RULES (MANDATORY)
==============================

Each feature MUST follow EXACTLY this pattern:

df.groupby('User')['COLUMN'].AGG()

OR:

df.groupby('User').size()

Where:
- COLUMN = exactly ONE column
- AGG MUST be one of: mean, sum, count, nunique, max, min, std

==============================
ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
==============================

- NEVER use df['User'].map()
- NEVER use .values
- NEVER use reset_index()
- NEVER use multiple columns (NO [['col1','col2']])
- NEVER use resample()
- NEVER use datetime operations
- NEVER return DataFrame (must return Series)
- NEVER perform arithmetic between multiple aggregations
- NEVER nest groupby
- NEVER use apply()
- NEVER chain column selection like: df.groupby(... )['col1']['col2']

==============================
VALID EXAMPLES (ONLY THESE TYPES)
==============================

df.groupby('User')['Amount'].mean()

df.groupby('User')['MCC'].nunique()

df.groupby('User')['Errors?'].sum()

df.groupby('User')['Hour'].std()

df.groupby('User').size()

==============================
INVALID EXAMPLES (DO NOT GENERATE)
==============================

df['User'].map(...)

df.groupby('User')[['Amount','Hour']]

df.groupby('User').resample('H')

df.groupby('User')['Amount'].mean().reset_index()

df.groupby('User')['Amount'].mean() / df.groupby('User').size()

==============================
FEATURE DESIGN FOCUS
==============================

- spending patterns (mean, std, max)
- transaction frequency (counts)
- merchant/category diversity (nunique)
- geographic spread (city/state/zip diversity)
- time behavior (Hour-based stats)
- error behavior
- chip usage patterns

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "type": "numeric"
  }
]

==============================
FINAL INSTRUCTIONS
==============================

- Generate 12–15 high-quality features
- Each feature MUST return a pandas Series indexed by User
- Length MUST equal number of unique users
- Do NOT compute values
- Do NOT explain anything
- Output ONLY JSON
- Output MUST start with '[' and end with ']'
- Use double quotes only
"""

In [13]:
# PROMPT="""You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# ---

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a RandomForestClassifier detect fraud.

# ---

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users

# ---

# FEATURE DESIGN FOCUS:

# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior (USE Hour only)
# - error behavior
# - chip usage patterns

# ---

# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features
# - using Time column
# - non-numeric outputs

# ---

# ==============================
# STRICT EXECUTION TEMPLATE (MANDATORY)
# ==============================

# Every feature MUST follow EXACTLY ONE of these two forms:

# FORM 1:
# df['User'].map(df.groupby('User')['COLUMN'].AGG())

# FORM 2:
# df['User'].map(df.groupby('User').size())

# Where:
# - COLUMN = exactly ONE column name (string)
# - AGG MUST be one of:
#   mean, sum, count, nunique, max, min, std

# ==============================
# ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
# ==============================

# - NEVER use .values
# - NEVER use reset_index()
# - NEVER use multiple columns (NO [['col1','col2']])
# - NEVER use resample()
# - NEVER use datetime operations
# - NEVER return DataFrame
# - NEVER perform arithmetic between two map() calls
# - NEVER nest groupby operations
# - NEVER use apply()
# - NEVER chain column selection like:
#   df.groupby(... )['col1']['col2']

# - You may ONLY select ONE column after groupby:
#   df.groupby('User')['COLUMN']

# ==============================
# VALID EXAMPLES (ONLY THESE TYPES)
# ==============================

# df['User'].map(df.groupby('User')['Amount'].mean())

# df['User'].map(df.groupby('User')['MCC'].nunique())

# df['User'].map(df.groupby('User')['Errors?'].sum())

# df['User'].map(df.groupby('User').size())

# ==============================
# INVALID EXAMPLES (DO NOT GENERATE)
# ==============================

# df.groupby('User').size().values

# df.groupby('User')[['Amount','Hour']]

# df.groupby('User').resample('H')

# df['User'].map(A) / df['User'].map(B)

# df.groupby('User')['Amount'].mean().reset_index()



# EXAMPLE (FOLLOW THIS EXACT STYLE):

# {
#   "feature_name": "avg_amount_per_user",
#   "pandas_code": "df['User'].map(df.groupby('User')['Amount'].mean())",
#   "description": "Average transaction amount per user",
#   "type": "numeric"
# }

# ---

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# ---

# FINAL INSTRUCTIONS:

# - Generate 12–15 high-quality fraud detection features
# - Do NOT compute values
# - Do NOT explain anything
# - Do NOT output anything outside JSON
# - Ensure VALID JSON (double quotes only)
# ==============================
# FINAL REQUIREMENT
# ==============================

# If a feature cannot be written EXACTLY in one of the VALID forms above,
# DO NOT include it.
# """

In [14]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [15]:
# response = call_llm(PROMPT)

response = model_llm.generate_content(PROMPT)
print("RAW OUTPUT:\n", response)

RAW OUTPUT:
 response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "[\n  {\n    \"feature_name\": \"user_transaction_count\",\n    \"pandas_code\": \"df.groupby('User').size()\",\n    \"description\": \"Total number of transactions per user.\",\n    \"type\": \"numeric\"\n  },\n  {\n    \"feature_name\": \"user_avg_amount\",\n    \"pandas_code\": \"df.groupby('User')['Amount'].mean()\",\n    \"description\": \"Average transaction amount per user.\",\n    \"type\": \"numeric\"\n  },\n  {\n    \"feature_name\": \"user_std_amount\",\n    \"pandas_code\": \"df.groupby('User')['Amount'].std()\",\n    \"description\": \"Standard deviation of transaction amounts per user (volatility of spending).\",\n    \"type\": \"numeric\"\n  },\n  {\n    \"feature_name\": \"user_total_errors\",\n    \"pandas_code\": \"df.groupby('U

In [16]:
import re

def fix_code(code):
    # Fix missing quotes
    replacements = {
        "[Amount]": "['Amount']",
        "[Time]": "['Time']",
        "[Is Fraud?]": "['Is Fraud?']"
    }
    for k, v in replacements.items():
        code = code.replace(k, v)

    # 🔥 Fix tuple column selection → list
    code = re.sub(
        r"\[['\"](\w+)['\"],\s*['\"](\w+)['\"]\]",
        r"[[ '\1', '\2' ]]",
        code
    )

    # Also fix ('A','B') → ['A','B']
    code = re.sub(
        r"\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)",
        r"['\1', '\2']",
        code
    )

    return code

In [17]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    


In [18]:
import re

ALLOWED_AGGS = ["mean", "sum", "count", "nunique", "max", "min", "std"]

def fix_code(code):
    if not code:
        return None

    code = code.strip()

    # =========================
    # 1. Fix missing quotes for columns
    # =========================
    code = re.sub(r"\[(\w[\w\s?]*)\]", r"['\1']", code)

    # =========================
    # 2. Fix tuple → list
    # =========================
    code = re.sub(r"\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)", r"['\1', '\2']", code)

    # =========================
    # 3. Normalize spaces
    # =========================
    code = re.sub(r"\s+", " ", code)

    # =========================
    # 4. MUST contain correct groupby
    # =========================
    if "df.groupby('User')" not in code:
        return None

    # =========================
    # 5. Reject multi-column groupby
    # =========================
    if "groupby(['User'" in code or 'groupby(["User"' in code:
        return None

    # =========================
    # 6. Reject multiple groupby
    # =========================
    if code.count("groupby") > 1:
        return None

    # =========================
    # 7. Reject forbidden ops
    # =========================
    forbidden = [
        "reset_index", "apply", "map", "values",
        "resample", "[[", "]]"
    ]
    if any(f in code for f in forbidden):
        return None

    # =========================
    # 8. Reject arithmetic (multiple aggregations)
    # =========================
    if any(op in code for op in ["/", "+", "-", "*"]):
        return None

    # =========================
    # 9. Validate aggregation
    # =========================
    agg_match = re.search(r"\.(\w+)\(\)", code)
    if not agg_match:
        return None

    agg = agg_match.group(1)
    if agg not in ALLOWED_AGGS and "size()" not in code:
        return None

    # =========================
    # 10. Final strict pattern check
    # =========================
    pattern1 = r"^df\.groupby\('User'\)\['[^']+'\]\.(mean|sum|count|nunique|max|min|std)\(\)$"
    pattern2 = r"^df\.groupby\('User'\)\.size\(\)$"

    if not (re.match(pattern1, code) or re.match(pattern2, code)):
        return None

    return code

In [19]:
# # response = model_llm.generate_content(PROMPT)
# features = extract_json(response)


# user_df = pd.DataFrame()

# user_df = pd.DataFrame(index=df['User'].unique())

# for f in features:
#     try:
#         code = fix_code(f.get("pandas_code", ""))

#         if not code:
#             print("Skipping invalid:", f["feature_name"])
#             continue

#         print("Applying:", f["feature_name"])

#         result = eval(code)

#         if not isinstance(result, pd.Series):
#             raise ValueError("Not a pandas Series")

#         result = result.reindex(user_df.index)

#         user_df[f["feature_name"]] = result
        
#     except Exception as e:
#         print("Error in", f["feature_name"], ":", e)
        

# user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()
# user_df["is_fraud_user"] = user_df["is_fraud_user"].reindex(user_df.index)

# user_df = user_df.fillna(0)

# # for f in features:
# #     try:
# #         print("Applying:", f["feature_name"])
# #         user_df[f["feature_name"]] = eval(f["pandas_code"])
# #     except Exception as e:
# #         print("Error in", f["feature_name"], ":", e)

# # user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# # user_df = user_df.fillna(0)



In [20]:
response = model_llm.generate_content(PROMPT)

output = response.text
# print("Raw Output:\n", output)

output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Applying: user_avg_transaction_amount
Applying: user_std_transaction_amount
Applying: user_transaction_count
Applying: user_merchant_diversity
Applying: user_city_diversity
Applying: user_state_diversity
Applying: user_zip_std
Applying: user_avg_hour
Applying: user_std_hour
Applying: user_error_count
Applying: user_chip_usage_rate
Applying: user_max_transaction_amount
Applying: user_mcc_diversity

Final Shape: (2000, 14)
      user_avg_transaction_amount  user_std_transaction_amount  \
User                                                             
0                       81.299989                    94.159093   
1                       81.118050                   156.784691   
2                       35.159687                    54.298552   
3                      117.277603                   268.654472   
4                       97.011698                   138.619564   

      user_transaction_count  user_merchant_diversity  user_city_diversity  \
User                              

In [21]:
print("\nFinal Shape:", user_df.shape)
print(user_df.head())


Final Shape: (2000, 14)
      user_avg_transaction_amount  user_std_transaction_amount  \
User                                                             
0                       81.299989                    94.159093   
1                       81.118050                   156.784691   
2                       35.159687                    54.298552   
3                      117.277603                   268.654472   
4                       97.011698                   138.619564   

      user_transaction_count  user_merchant_diversity  user_city_diversity  \
User                                                                         
0                      19963                      552                  295   
1                       8919                      337                  263   
2                      41978                      727                  314   
3                      10117                      655                  392   
4                      18542                

In [22]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [23]:
print(user_df.shape)

(2000, 14)


In [24]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [25]:
print(y.isna().sum())

0


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.90      0.70      0.79       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.89      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400

ROC-AUC: 0.904452453247822


In [29]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

user_transaction_count         0.172261
user_error_count               0.171005
user_state_diversity           0.130216
user_mcc_diversity             0.123085
user_merchant_diversity        0.123027
user_city_diversity            0.100350
user_max_transaction_amount    0.054801
user_std_transaction_amount    0.044237
user_zip_std                   0.041354
user_avg_transaction_amount    0.039664
user_avg_hour                  0.000000
user_std_hour                  0.000000
user_chip_usage_rate           0.000000
dtype: float64


In [30]:
feature_names = [f["feature_name"] for f in features]

In [31]:
top_features = importance.to_dict()

In [32]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}


In [33]:
print(report1)
print(importance)
print(roc1)

              precision    recall  f1-score   support

           0       0.90      0.70      0.79       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.89      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400

user_transaction_count         0.172261
user_error_count               0.171005
user_state_diversity           0.130216
user_mcc_diversity             0.123085
user_merchant_diversity        0.123027
user_city_diversity            0.100350
user_max_transaction_amount    0.054801
user_std_transaction_amount    0.044237
user_zip_std                   0.041354
user_avg_transaction_amount    0.039664
user_avg_hour                  0.000000
user_std_hour                  0.000000
user_chip_usage_rate           0.000000
dtype: float64
0.904452453247822


In [ ]:
sample = df.sample(n=5, random_state=42)
print(sample)
sample_str = sample.to_string(index=False)

   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Verne             CA  91750.0  5300        1          0   0.0  
1  Monterey Park             CA  91754.0  5411        1          0   0.0  
2  Monterey Park             CA  91754.0  5411        1          0   0.0  
3  Monterey Park             CA  91754.0  5651        1          0   0.0  
4       La Verne             CA  91750.0  5912        1          0   0.0  


In [35]:
def get_base_features(features):
    base_features = []

    for f in features:
        code = f["pandas_code"]

        # Only keep df-based features
        if "df.groupby" in code and "user_" not in code.split("=")[-1]:
            base_features.append(f)

    return base_features


In [36]:
def apply_base_features(features):
    user_df = pd.DataFrame(index=df['User'].unique())

    for f in features:
        try:
            print("Applying:", f["feature_name"])
            
            result = eval(f["pandas_code"])
            
            # ---- VALIDATION ----
            if not isinstance(result, pd.Series):
                raise ValueError("Not a pandas Series")
            
            # ---- FORCE ALIGNMENT ----
            result = result.reindex(user_df.index)
            
            user_df[f["feature_name"]] = result
            
        except Exception as e:
            print("Error in", f["feature_name"], ":", e)
            

    user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()
    user_df["is_fraud_user"] = user_df["is_fraud_user"].reindex(user_df.index)

    user_df = user_df.fillna(0)

In [37]:
def split_data(user_df):
    X = user_df.drop(columns=["is_fraud_user"])
    y = user_df["is_fraud_user"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    return X, y, X_train, X_test, y_train, y_test

In [38]:
def train_model(X_train, y_train):
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred))

    print("\n=== ROC AUC ===")
    print(roc_auc_score(y_test, y_prob))

    return y_pred, y_prob
def get_feature_importance(model, X):
    
    importance = pd.Series(model.feature_importances_, index=X.columns)
    importance = importance.sort_values(ascending=False)

    print("\n=== Top Features ===")
    print(importance.head(10))

    return importance

In [ ]:
# def get_prompt(llm_input):
#     PROMPT = f"""
# You are a fraud feature engineering expert.

# A RandomForest model was trained on USER-level aggregated features.

# ==============================
# DATA (IMPORTANT)
# ==============================

# User (group key), Year, Month, Day, Hour (0–23), Amount, Use Chip (0/1),
# Merchant City, Merchant State, Zip, MCC, Errors? (0/1)

# DO NOT USE: Card, Time, Merchant Name, Is Fraud

# ==============================
# CURRENT FEATURES
# ==============================
# {llm_input["existing_features"]}

# ==============================
# MODEL PERFORMANCE
# ==============================
# {llm_input["classification_report"]}
# ROC-AUC: {llm_input["roc_auc"]}
# Top Features: {llm_input["top_features"]}

# ==============================
# TASK
# ==============================

# Suggest 20-30 NEW user-level features to improve fraud recall.

# Focus:
# - spending patterns (Amount)
# - transaction volume
# - merchant/category diversity
# - geographic spread
# - time behavior (Hour)
# - errors
# - chip usage

# ==============================
# STRICT RULES (MANDATORY)
# ==============================

# Each feature MUST be EXACTLY one of:

# df.groupby('User')['COLUMN'].AGG()

# OR

# df.groupby('User').size()

# Where:
# - COLUMN = ONE column only
# - AGG ∈ [mean, sum, count, nunique, max, min, std]

# ==============================
# FORBIDDEN
# ==============================

# - Multiple columns
# - Multiple aggregations
# - Arithmetic between aggregations
# - apply(), map(), reset_index(), values
# - datetime / resample
# - nested groupby
# - using other features

# If violated → feature is INVALID.

# ==============================
# OUTPUT FORMAT (STRICT JSON)
# ==============================

# [
#   {{
#     "feature_name": "...",
#     "pandas_code": "df.groupby('User')['COLUMN'].AGG()",
#     "description": "...",
#     "type": "numeric"
#   }}
# ]

# ==============================
# CRITICAL
# ==============================

# - Output ONLY JSON
# - No text before/after
# - No explanations
# - Must start with [ and end with ]
# - Must be valid json.loads()

# If not → answer is WRONG.
# """
#     return PROMPT

In [74]:
def get_prompt(llm_input, df):
    sample = df.sample(n=5, random_state=42)
    sample_str = sample.to_string(index=False)

    PROMPT = f"""
You are a fraud analytics feature engineering expert.

A RandomForestClassifier was trained on USER-LEVEL features.

==============================
DATA SAMPLE (RANDOM)
==============================
{sample_str}

==============================
MODEL PERFORMANCE
==============================

Classification Report:
{llm_input["classification_report"]}

ROC-AUC:
{llm_input["roc_auc"]}

Top Important Features:
{llm_input["top_features"]}

==============================
DATASET SCHEMA (STRICT)
==============================

- User → int64 (group key)
- Card → int64 (DO NOT USE)
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (DO NOT USE)
- Hour → float64 (USE THIS)
- Amount → float64
- Use Chip → float64
- Merchant Name → int64 (DO NOT USE)
- Merchant City → object (ONLY nunique/count)
- Merchant State → object (ONLY nunique/count)
- Zip → float64
- MCC → int64
- Errors? → int64
- Is Fraud? → int64 (DO NOT USE)

==============================
CRITICAL GOAL
==============================

Generate USER-LEVEL features.

Each feature MUST produce:
👉 EXACTLY ONE VALUE PER USER

==============================
STRICT EXECUTION RULES (HARD CONSTRAINTS)
==============================

Every feature MUST follow EXACTLY ONE of these:

1.
df.groupby('User')['COLUMN'].AGG()

OR

2.
df.groupby('User').size()

Where:
- COLUMN = exactly ONE column
- AGG ∈ [mean, sum, count, nunique, max, min, std]

🚨 ANY deviation is INVALID.

==============================
ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
==============================

- NO df['User'].map()
- NO reset_index()
- NO apply()
- NO multiple columns
- NO nested aggregation
- NO arithmetic between aggregations
- NO resample
- NO datetime operations
- NO referencing other features
- NO use of Time
- NO Merchant Name usage
- NO DataFrame output (ONLY Series)

==============================
INVALID PATTERNS (STRICTLY FORBIDDEN)
==============================

df.groupby('User')['Amount'].mean().mean()
df.groupby('User')['Amount'].mean() / df.groupby('User').size()
df.groupby('User')[['Amount','Hour']]
df.groupby('User').apply(...)
df['User'].map(...)

==============================
YOUR TASK
==============================

1. Identify missing behavioral signals from model performance
2. Focus on improving fraud recall
3. Generate 50 HIGH-QUALITY USER-LEVEL features

==============================
FEATURE DESIGN FOCUS
==============================

- spending variability (std, max, min)
- transaction bursts (counts)
- merchant/category diversity (nunique)
- geographic spread
- hour-based behavior
- error patterns
- chip usage patterns

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

[
  {{
    "feature_name": "...",
    "pandas_code": "...",
    "reason": "..."
  }}
]

==============================
FINAL RULES
==============================

- EVERY pandas_code MUST start with:
  df.groupby('User')

- EVERY output MUST be:
  pandas Series indexed by User

- If ANY feature violates rules → DO NOT include it

- DO NOT explain anything outside JSON

- OUTPUT ONLY JSON
"""
    return PROMPT

In [75]:
def clean_features(X):
    # Replace inf → NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # Fill NaN with 0 (or better: median)
    X = X.fillna(0)

    # Optional: clip extreme values (very important)
    X = X.clip(-1e10, 1e10)

    return X

In [76]:
def get_userdf(llm_input):
    PROMPT = get_prompt(llm_input,df)

    response = model_llm.generate_content(PROMPT)

    output = response.text
    # print("Raw Output:\n", output)

    output = output.replace("```json", "").replace("```", "").strip()

    features = json.loads(output)


    user_df = pd.DataFrame()

    for f in features:
        try:
            print("Applying:", f["feature_name"])
            user_df[f["feature_name"]] = eval(f["pandas_code"])
        except Exception as e:
            print("Error in", f["feature_name"], ":", e)

    # ✅ Add label
    user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

    # ✅ Fill missing
    user_df = user_df.fillna(0)
    return user_df
    

In [77]:


def run(user_df):
    
    X, y, X_train, X_test, y_train, y_test = split_data(user_df)
    X = clean_features(X)

    model = train_model(X_train, y_train)

    print("\n=== Model Evaluation ===")
    y_pred, y_prob = evaluate_model(model, X_test, y_test)
    
    report1 = classification_report(y_test, y_pred)
    roc1 = roc_auc_score(y_test, y_prob)
    importance = get_feature_importance(model, X)

    # print("\n=== Top Features ===")
    # print(importance)
    top_features = importance.to_dict()
    feature_names = [f["feature_name"] for f in features]
    
    llm_input = {
        "existing_features": feature_names,
        "classification_report": report1,
        "roc_auc": roc1,
        "top_features": top_features
    }
    return llm_input


In [78]:
user_df=get_userdf(llm_input)


Applying: user_amount_mean
Applying: user_amount_median
Applying: user_amount_std
Applying: user_amount_max
Applying: user_amount_min
Applying: user_transaction_count
Applying: user_error_sum
Applying: user_error_rate
Applying: user_chip_usage_sum
Applying: user_chip_usage_rate
Applying: user_hour_mean
Applying: user_hour_std
Applying: user_year_nunique
Applying: user_month_nunique
Applying: user_day_nunique
Applying: user_mcc_nunique
Applying: user_zip_nunique
Applying: user_state_nunique
Applying: user_city_nunique
Applying: user_amount_abs_diff_mean
Applying: user_amount_max_div_mean
Applying: user_amount_range
Applying: user_transactions_per_year
Applying: user_error_per_transaction
Applying: user_chip_non_usage_rate
Applying: user_hour_diff_std
Applying: user_mcc_entropy
Applying: user_state_entropy
Applying: user_zip_entropy
Applying: user_time_since_last_tx
Applying: user_error_max
Applying: user_first_year
Applying: user_last_year
Applying: user_count_morning
Applying: user_cou

In [82]:
print(user_df.shape)
llm_input=run(user_df)

print("\n===== LLM INPUT =====")

print("\nExisting Features:")
print(llm_input["existing_features"])

print("\nROC-AUC:")
print(llm_input["roc_auc"])

print("\nTop Features:")
for k, v in llm_input["top_features"].items():
    print(f"{k}: {v}")

print("\nClassification Report:")
print(llm_input["classification_report"])


(2000, 50)

=== Model Evaluation ===

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.91      0.71      0.80       131
           1       0.87      0.97      0.92       269

    accuracy                           0.88       400
   macro avg       0.89      0.84      0.86       400
weighted avg       0.89      0.88      0.88       400


=== ROC AUC ===
0.9157467578535146

=== Top Features ===
user_year_nunique         0.115802
user_count_night          0.087359
user_first_year           0.073873
user_zip_count            0.072266
user_transaction_count    0.072179
user_error_sum            0.062436
user_mcc_nunique          0.053281
user_zip_nunique          0.046775
user_state_nunique        0.043132
user_city_nunique         0.038938
dtype: float64

===== LLM INPUT =====

Existing Features:
['user_median_transaction_amount', 'user_pct_negative_amount', 'user_max_hour_gap', 'user_total_error_count', 'user_chip_use_count', 'user_

In [80]:
# run(llm_input)
